# Single Modality Analysis of Mouse Thymus Dataset

## Loading Packages

In [ ]:
import warnings
warnings.filterwarnings('ignore')
## Loading package
import os

import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

# the location of R (used for the mclust clustering)
os.environ['R_HOME'] = 'E:/R-4.3.1'
os.environ['R_USER'] = 'E:/anaconda/lib/site-packages/rpy2'

save_path = 'D:/study/learning/spatial_transcriptome/papers/spatial_multi_omics-main/Results/Visualization/Mouse_Thymus/' 

import sys
sys.path.insert(0, 'D:/study/learning/spatial_transcriptome/papers/spatial_multi_omics-main/Model/')
from preprocess import preprocessing
from utils import mclust_R

## Loading and Preprocessing Data

In [ ]:
file_fold_1 = 'D:/study/learning/spatial_transcriptome/papers/spatial_multi_omics-main/data/Mouse_Thymus_1/'
file_fold_2 = 'D:/study/learning/spatial_transcriptome/papers/spatial_multi_omics-main/data/Mouse_Thymus_2/'
file_fold_3 = 'D:/study/learning/spatial_transcriptome/papers/spatial_multi_omics-main/data/Mouse_Thymus_3/'

adata_omics_1_1 = sc.read_h5ad(file_fold_1 + 'adata_RNA.h5ad')
adata_omics_1_2 = sc.read_h5ad(file_fold_1 + 'adata_ADT.h5ad')

adata_omics_2_1 = sc.read_h5ad(file_fold_2 + 'adata_RNA.h5ad')
adata_omics_2_2 = sc.read_h5ad(file_fold_2 + 'adata_ADT.h5ad')

adata_omics_3_1 = sc.read_h5ad(file_fold_3 + 'adata_RNA.h5ad')
adata_omics_3_2 = sc.read_h5ad(file_fold_3 + 'adata_ADT.h5ad')

adata_omics_1_1.var_names_make_unique()
adata_omics_1_2.var_names_make_unique()
adata_omics_2_1.var_names_make_unique()
adata_omics_2_2.var_names_make_unique()
adata_omics_3_1.var_names_make_unique()
adata_omics_3_2.var_names_make_unique()

adata_omics_1_1, adata_omics_1_2 = preprocessing(adata_omics_1_1, adata_omics_1_2, 'Stereo-CITE-seq')
adata_omics_2_1, adata_omics_2_2 = preprocessing(adata_omics_2_1, adata_omics_2_2, 'Stereo-CITE-seq')
adata_omics_3_1, adata_omics_3_2 = preprocessing(adata_omics_3_1, adata_omics_3_2, 'Stereo-CITE-seq')

Stereo-CITE-seq data preprocessing have done!
Dimensions after preprocessed adata_modal_1: (4183, 3000)
Dimensions after preprocessing adata_modal_2: (4183, 19)
Stereo-CITE-seq data preprocessing have done!
Dimensions after preprocessed adata_modal_1: (4573, 3000)
Dimensions after preprocessing adata_modal_2: (4573, 19)
Stereo-CITE-seq data preprocessing have done!
Dimensions after preprocessed adata_modal_1: (4147, 3000)
Dimensions after preprocessing adata_modal_2: (4147, 19)


In [ ]:
adata_RNA_analysis = adata_omics_3_1
adata_Protein_analysis = adata_omics_3_2

In [ ]:
## Running PCA on RNA modality
sc.pp.pca(adata_RNA_analysis, n_comps=adata_Protein_analysis.shape[1]-1)
sc.pp.neighbors(adata_RNA_analysis, use_rep='X_pca')
sc.tl.umap(adata_RNA_analysis)
mclust_R(adata_RNA_analysis, used_obsm='X_pca', num_cluster=7)

## Running PCA on Protein modality
sc.pp.pca(adata_Protein_analysis, n_comps=adata_Protein_analysis.shape[1]-1)
sc.pp.neighbors(adata_Protein_analysis, use_rep='X_pca')
sc.tl.umap(adata_Protein_analysis)
mclust_R(adata_Protein_analysis, used_obsm='X_pca', num_cluster=7)

fitting ...
  |======================================================================| 100%
fitting ...
  |======================================================================| 100%


AnnData object with n_obs × n_vars = 4147 × 19
    obs: 'orig.ident', 'x', 'y', 'clusters_mclust'
    var: 'n_cells'
    uns: 'INR', 'pca', 'neighbors', 'umap'
    obsm: 'spatial', 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

## Storing the Results

In [79]:
results = sc.read_h5ad('D:/study/learning/spatial_transcriptome/papers/spatial_multi_omics-main/Results/Mouse_Thymus_Replicate3.h5ad')
results

AnnData object with n_obs × n_vars = 4147 × 0
    obs: 'SpaKnit', 'SpatialGlue', 'STAGATE', 'Modality1', 'Modality2'
    obsm: 'Modality1', 'Modality2', 'STAGATE', 'SpaKnit', 'SpatialGlue', 'spatial'

In [73]:
results.obs['Modality1'] = adata_RNA_analysis.obs['clusters_mclust'].values
results.obsm['Modality1'] = adata_RNA_analysis.obsm['X_pca']
results.obs['Modality2'] = adata_Protein_analysis.obs['clusters_mclust'].values
results.obsm['Modality2'] = adata_Protein_analysis.obsm['X_pca']
results.write_h5ad('D:/study/learning/spatial_transcriptome/papers/spatial_multi_omics-main/Results/Mouse_Thymus_Replicate3.h5ad')

## Visualization

In [ ]:
colors = [
    '#2a9d8f', '#ee6055', '#fdf0d5', '#83c5be', '#99582a', '#f9c74f',  '#264653'
]

fig,ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))
sc.pl.embedding(adata_Protein_analysis, basis='spatial', color='clusters_mclust', ax=ax, s=100, show=False, palette=colors)
ax.invert_yaxis()
ax.set_title(f'')
ax.set_xlabel('')
ax.set_ylabel('')
# remove legend
ax.get_legend().remove()
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()

In [234]:
# # mapping = {1: 1, 2: 4, 3: 5, 4: 3, 5: 6, 6: 2}
# # adata_RNA_analysis.obs['clusters_mclust'] = adata_RNA_analysis.obs['clusters_mclust'].map(mapping)
# mapping = {1: 5, 2: 3, 3: 4, 4: 2, 5: 1, 6: 6}
# adata_Protein_analysis.obs['clusters_mclust'] = adata_Protein_analysis.obs['clusters_mclust'].map(mapping)

In [ ]:
# RNA umap visualization
plt.rcParams['font.size'] = 20
plt.rcParams['font.sans-serif'] = 'Arial'
fig, ax = plt.subplots(1,1, figsize=(6,6))

sc.pl.umap(adata_RNA_analysis, color='clusters_mclust', ax=ax,legend_loc='on data',legend_fontoutline=5, show=False)
ax.set_title('')
# remove x, y axis
ax.set_xlabel('')
ax.set_ylabel('')

# plt.savefig(save_path + 'thymus3_RNA_umap.png', bbox_inches='tight', dpi=500)
# plt.savefig(save_path + 'thymus3_RNA_umap.eps', bbox_inches='tight', dpi=500)

In [ ]:
# # RNA PAGA
plt.rcParams['font.size'] = 20
plt.rcParams['font.sans-serif'] = 'Arial'
fig, ax = plt.subplots(1,1, figsize=(6,6))
sc.tl.paga(adata_RNA_analysis, groups='clusters_mclust')
sc.pl.paga(adata_RNA_analysis, edge_width_scale=1, node_size_scale=5, ax=ax, show=False, threshold=0.1, fontoutline=3)
# ax.set_title('PAGA graph')
# remove x, y axis
ax.set_xlabel('')
ax.set_ylabel('')

# plt.savefig(save_path + 'thymus3_RNA_PAGA.png', bbox_inches='tight', dpi=500)
# plt.savefig(save_path + 'thymus3_RNA_PAGA.eps', bbox_inches='tight', dpi=500)

In [ ]:
plt.rcParams['font.size'] = 20
plt.rcParams['font.sans-serif'] = 'Arial'
fig, ax = plt.subplots(1,1, figsize=(6,6))

sc.pl.umap(adata_Protein_analysis, color='clusters_mclust', ax=ax,legend_loc='on data',legend_fontoutline=5, show=False)
ax.set_title('')
# remove x, y axis
ax.set_xlabel('')
ax.set_ylabel('')

# plt.savefig(save_path + 'thymus3_Protein_umap.png', bbox_inches='tight', dpi=500)
# plt.savefig(save_path + 'thymus3_Protein_umap.eps', bbox_inches='tight', dpi=500)

In [ ]:
plt.rcParams['font.size'] = 20
plt.rcParams['font.sans-serif'] = 'Arial'
fig, ax = plt.subplots(1,1, figsize=(6,6))
sc.tl.paga(adata_Protein_analysis, groups='clusters_mclust')
sc.pl.paga(adata_Protein_analysis, edge_width_scale=3, node_size_scale=5, ax=ax, show=False, threshold=0.1, fontoutline=3)
ax.set_title('')
# remove x, y axis
ax.set_xlabel('')
ax.set_ylabel('')

# plt.savefig(save_path + 'thymus3_Protein_PAGA.png', bbox_inches='tight', dpi=500)
# plt.savefig(save_path + 'thymus3_Protein_PAGA.eps', bbox_inches='tight', dpi=500)

In [ ]:
## sc.tl.rank_genes_groups: Genes are ranked to determine population characteristics
sc.tl.rank_genes_groups(adata_RNA_analysis, 'clusters_mclust', method="t-test")
rank_genes = sc.get.rank_genes_groups_df(adata_RNA_analysis, group=None)
rank_genes.to_excel(save_path + 'marker_genes.xlsx', index=True)

In [ ]:
plt.rcParams['font.size'] = 18
plt.rcParams['font.sans-serif'] = 'Arial'
fig, ax = plt.subplots(1,6, figsize=(24,4))
fig.subplots_adjust(wspace=0.1, hspace=0)
marker_genes = ['Themis', 'Rag1', 'Tmsb4x', 'Rag1', 'H2-K1', 'Gm26917']
# marker_genes = ['Cdk8', 'Rag1', 'Gm26917', 'Prss16', 'Arpp21', 'H2-K1']
# marker_genes = ['Cdk8', 'Apoe', 'H2-K1', 'Rag1', 'Cdk8', 'Rag1']
components_range =range(6)
for i in components_range:

    sc.pl.spatial(adata_omics_1_1, img_key=None, color=marker_genes[i], spot_size=100, show=False, ax=ax[i], colorbar_loc=None, cmap='coolwarm')
    # remove x, y axis
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')
    ax[i].set_aspect(1.55)
# plt.savefig(save_path + 'thymus1_Marker.png', bbox_inches='tight', dpi=500)
# plt.savefig(save_path + 'thymus1_Marker.eps', bbox_inches='tight', dpi=500)
plt.show()

In [ ]:
plt.rcParams['font.size'] = 16
plt.rcParams['font.sans-serif'] = 'Arial'
fig, ax = plt.subplots(1,1, figsize=(10,8))
ax = sc.pl.stacked_violin(adata_RNA_analysis, marker_genes, groupby='clusters_mclust', figsize=(15,8), ax=ax,dendrogram=True, show=False)
# plt.savefig(save_path + 'thymus3_Marker_violin.png', bbox_inches='tight', dpi=500)
# plt.savefig(save_path + 'thymus3_Marker_violin.eps', bbox_inches='tight', dpi=500)
plt.show()

In [ ]:
# adata_RNA_analysis.write_h5ad(save_path + 'thymus3_RNA_Results.h5ad')
# adata_Protein_analysis.write_h5ad(save_path + 'thymus3_Protein_Results.h5ad')